In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import sys
sys.path.append('/content/drive/MyDrive/Colab Notebooks')  # adjust to your exact folder name

from Data_Utils import prepare_2016_dataset

splits = prepare_2016_dataset(
    pkl_path='/content/drive/MyDrive/Colab Notebooks/RML2016.10a_dict.pkl',
    save_path='/content/drive/MyDrive/Colab Notebooks/radioml2016_prepped.npz'
)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import sys
sys.path.append('/content/drive/MyDrive/Colab Notebooks')

from Data_Utils import load_prepped
from model import AMCTransformer
from train import train_model

splits, class_names = load_prepped('/content/drive/MyDrive/Colab Notebooks/radioml2016_prepped.npz')
print("Classes:", class_names)

model, history = train_model(
    splits,
    num_classes=len(class_names),
    seq_len=128,
    epochs=30,
    checkpoint_path='/content/drive/MyDrive/Colab Notebooks/amc_transformer_stage1.pt'
)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import sys
sys.path.append('/content/drive/MyDrive/Colab Notebooks')

In [ ]:
from data_utils import load_prepped
from evaluate import evaluate_model

splits, class_names = load_prepped('/content/drive/MyDrive/Colab Notebooks/radioml2016_prepped.npz')

results = evaluate_model(
    checkpoint_path='/content/drive/MyDrive/Colab Notebooks/amc_transformer_stage1.pt',
    splits=splits,
    class_names=list(class_names),
    seq_len=128,
    save_dir='/content/drive/MyDrive/Colab Notebooks'
)

In [ ]:
import os
os.environ['KAGGLE_USERNAME'] = 'enter yours'
os.environ['KAGGLE_KEY'] = '****'

In [ ]:
!pip install kaggle -q
!kaggle datasets download -d pinxau1000/radioml2018 -p /content/radioml2018
!unzip -q /content/radioml2018/radioml2018.zip -d /content/radioml2018
!ls -lh /content/radioml2018/

In [ ]:
!ls -la "/content/drive/MyDrive/Colab Notebooks/data_utils.py"

In [ ]:
import sys, os

folder = '/content/drive/MyDrive/Colab Notebooks'
print("Folder exists:", os.path.exists(folder))
print("Files in folder:", os.listdir(folder))
print("Is data_utils.py in there:", 'data_utils.py' in os.listdir(folder))
print()
print("Current sys.path:")
for p in sys.path:
    print(" ", repr(p))

In [ ]:
import importlib
importlib.invalidate_caches()

import data_utils
print(data_utils.RADIOML2018_CLASS_ORDER[:5])

In [ ]:
import os
from data_utils import prepare_2018_dataset

hdf5_path = '/content/radioml2018/GOLD_XYZ_OSC.0001_1024.hdf5'

if not os.path.exists(hdf5_path):
    print(f"Error: The HDF5 file does not exist at {hdf5_path}.")
    print("Please ensure that the dataset has been downloaded and unzipped correctly, and that the runtime has not been reset since.")
    print("You may need to re-run the cells for downloading and unzipping the dataset.")
else:
    print(f"HDF5 file found at {hdf5_path}.")
    splits_2018 = prepare_2018_dataset(
        hdf5_path=hdf5_path,
        save_path='/content/drive/MyDrive/Colab Notebooks/radioml2018_prepped.npz',
        max_samples_per_class=2000,
    )

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from model import AMCTransformer

@torch.no_grad()
def zero_shot_evaluate(checkpoint_path, splits, class_names, seq_len, patch_size=4, device=None):
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")

    model = AMCTransformer(seq_len=seq_len, num_classes=len(class_names), patch_size=patch_size)
    ckpt = torch.load(checkpoint_path, map_location=device)
    source_state = ckpt["model_state_dict"]
    filtered_state = {
        k: v for k, v in source_state.items()
        if k in model.state_dict() and model.state_dict()[k].shape == v.shape
    }
    skipped = set(source_state.keys()) - set(filtered_state.keys())
    print(f"Loaded {len(filtered_state)}/{len(source_state)} tensors from checkpoint.")
    print(f"Skipped (shape mismatch, randomly initialized): {skipped}")

    model.load_state_dict(filtered_state, strict=False)
    model = model.to(device).eval()

    X_test = torch.from_numpy(splits["X_test"]).float()
    y_test = splits["y_test"]
    snr_test = splits["snr_test"]

    preds = []
    batch_size = 512
    for i in range(0, len(X_test), batch_size):
        batch = X_test[i:i+batch_size].to(device)
        logits = model(batch)
        preds.append(logits.argmax(dim=1).cpu().numpy())
    y_pred = np.concatenate(preds)

    overall_acc = (y_pred == y_test).mean()
    print(f"\nZero-shot (no fine-tuning) overall test accuracy: {overall_acc:.4f}")

    print("\nZero-shot accuracy by SNR:")
    snr_acc = {}
    for s in sorted(np.unique(snr_test)):
        mask = snr_test == s
        acc = (y_pred[mask] == y_test[mask]).mean()
        snr_acc[int(s)] = acc
        print(f"  {s:+4d} dB: {acc:.4f}")

    plt.figure(figsize=(8,5))
    plt.plot(list(snr_acc.keys()), list(snr_acc.values()), marker='o')
    plt.axhline(y=1/len(class_names), color='gray', linestyle='--', label='random guess')
    plt.xlabel("SNR (dB)")
    plt.ylabel("Accuracy")
    plt.title("Zero-Shot: Stage 1 model on 2018.01A (no fine-tuning)")
    plt.ylim(0,1)
    plt.grid(alpha=0.3)
    plt.legend()
    plt.savefig('/content/drive/MyDrive/Colab Notebooks/zero_shot_snr_accuracy.png', dpi=150, bbox_inches='tight')
    plt.show()

    return {"overall_acc": overall_acc, "snr_acc": snr_acc, "y_true": y_test, "y_pred": y_pred}

In [ ]:
from data_utils import load_prepped

splits, class_names = load_prepped('/content/drive/MyDrive/Colab Notebooks/radioml2018_prepped.npz')
print("Classes:", class_names)

zero_shot_results = zero_shot_evaluate(
    checkpoint_path='/content/drive/MyDrive/Colab Notebooks/amc_transformer_stage1.pt',
    splits=splits,
    class_names=list(class_names),
    seq_len=1024,
    patch_size=4,
)

In [ ]:
import torch
from model import AMCTransformer
from train import train_model
from data_utils_2 import load_prepped

DRIVE_DIR = '/content/drive/MyDrive/Colab Notebooks'

# --- Load the prepped RadioML2018 data ---
splits, class_names = load_prepped(f'{DRIVE_DIR}/radioml2018_prepped.npz')
print("Classes:", class_names)

seq_len = splits['X_train'].shape[-1]   # 1024
num_classes = len(class_names)          # 7
patch_size = 4                           # same as stage1, so patch_embed weights transfer

# --- Build a fresh model sized for the new seq_len ---
model = AMCTransformer(seq_len=seq_len, num_classes=num_classes, patch_size=patch_size)

# --- Load stage1 checkpoint, skipping the shape-mismatched positional embedding ---
ckpt = torch.load(f'{DRIVE_DIR}/amc_transformer_stage1.pt', map_location='cpu')
stage1_state = ckpt['model_state_dict']

filtered_state = {
    k: v for k, v in stage1_state.items()
    if k in model.state_dict() and model.state_dict()[k].shape == v.shape
}
skipped = set(stage1_state.keys()) - set(filtered_state.keys())
print(f"Loaded {len(filtered_state)}/{len(stage1_state)} tensors from stage1.")
print(f"Skipped (shape mismatch, randomly initialized): {skipped}")

model.load_state_dict(filtered_state, strict=False)

# --- Freeze encoder + patch_embed, but keep pos_encoding/cls_token trainable ---
model.freeze_encoder(freeze=True)
model.pos_encoding.pos_embedding.requires_grad = True
model.cls_token.requires_grad = True

# --- Partial fine-tuning: unfreeze the last 2 encoder layers ---
model.unfreeze_last_n_layers(n=2)

# Sanity check: confirm which params are trainable
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {trainable:,} / {total:,}")

# --- Stage 2 fine-tuning run ---
model, history = train_model(
    splits,
    num_classes=num_classes,
    seq_len=seq_len,
    patch_size=patch_size,
    epochs=15,                 # fewer than stage1 — fine-tuning, not from-scratch
    lr=5e-5,                   # lower than stage1's 3e-4
    checkpoint_path=f'{DRIVE_DIR}/amc_transformer_stage2.pt',  # separate file, saved to Drive
    model=model,
)

In [ ]:
import torch
from model import AMCTransformer
from train import train_model

DRIVE_DIR = '/content/drive/MyDrive/Colab Notebooks'

seq_len = splits['X_train'].shape[-1]
num_classes = len(class_names)
patch_size = 4

model = AMCTransformer(seq_len=seq_len, num_classes=num_classes, patch_size=patch_size)

ckpt = torch.load(f'{DRIVE_DIR}/amc_transformer_stage1.pt', map_location='cpu')
stage1_state = ckpt['model_state_dict']
filtered_state = {k: v for k, v in stage1_state.items() if k in model.state_dict() and model.state_dict()[k].shape == v.shape}
model.load_state_dict(filtered_state, strict=False)

model.freeze_encoder(freeze=True)
model.pos_encoding.pos_embedding.requires_grad = True
model.cls_token.requires_grad = True
model.unfreeze_last_n_layers(n=2)

model, history = train_model(
    splits, num_classes=num_classes, seq_len=seq_len, patch_size=patch_size,
    epochs=15, lr=5e-5,
    checkpoint_path=f'{DRIVE_DIR}/amc_transformer_stage2.pt',
    model=model,
)

In [ ]:
@torch.no_grad()
def evaluate_finetuned(model, splits, class_names, device=None):
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device).eval()
    X_test = torch.from_numpy(splits["X_test"]).float()
    y_test = splits["y_test"]
    snr_test = splits["snr_test"]

    preds = []
    for i in range(0, len(X_test), 512):
        batch = X_test[i:i+512].to(device)
        preds.append(model(batch).argmax(dim=1).cpu().numpy())
    y_pred = np.concatenate(preds)

    overall_acc = (y_pred == y_test).mean()
    print(f"Fine-tuned overall test accuracy: {overall_acc:.4f}")

    snr_acc = {}
    for s in sorted(np.unique(snr_test)):
        mask = snr_test == s
        snr_acc[int(s)] = (y_pred[mask] == y_test[mask]).mean()

    plt.figure(figsize=(8,5))
    plt.plot(list(zero_shot_results['snr_acc'].keys()), list(zero_shot_results['snr_acc'].values()), marker='o', label='Zero-shot (no fine-tuning)')
    plt.plot(list(snr_acc.keys()), list(snr_acc.values()), marker='s', label='Fine-tuned (Stage 2)')
    plt.axhline(y=1/len(class_names), color='gray', linestyle='--', label='random guess')
    plt.xlabel("SNR (dB)"); plt.ylabel("Accuracy")
    plt.title("Zero-shot vs Fine-tuned: 2018.01A")
    plt.ylim(0,1); plt.grid(alpha=0.3); plt.legend()
    plt.savefig('/content/drive/MyDrive/Colab Notebooks/zeroshot_vs_finetuned.png', dpi=150, bbox_inches='tight')
    plt.show()
    return snr_acc

snr_acc_stage2 = evaluate_finetuned(model, splits, class_names)

In [ ]:
def plot_confusion_matrix_inline(y_true, y_pred, class_names, title="Confusion Matrix", min_snr=None, snr=None, save_path=None):
    if min_snr is not None and snr is not None:
        mask = snr >= min_snr
        y_true, y_pred = y_true[mask], y_pred[mask]

    n_classes = len(class_names)
    cm = np.zeros((n_classes, n_classes), dtype=int)
    for t, p in zip(y_true, y_pred):
        cm[t, p] += 1
    cm_norm = cm / cm.sum(axis=1, keepdims=True).clip(min=1)

    fig, ax = plt.subplots(figsize=(7,6))
    im = ax.imshow(cm_norm, cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(n_classes)); ax.set_yticks(range(n_classes))
    ax.set_xticklabels(class_names, rotation=45, ha="right")
    ax.set_yticklabels(class_names)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
    ax.set_title(title)
    for i in range(n_classes):
        for j in range(n_classes):
            val = cm_norm[i,j]
            ax.text(j, i, f"{val:.2f}", ha="center", va="center",
                     color="white" if val > 0.5 else "black", fontsize=8)
    fig.colorbar(im, ax=ax, label="Fraction of true class")
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"Saved to {save_path}")
    plt.show()

In [ ]:
# Need y_pred for Stage 2 model — reuse the predictions from evaluate_finetuned, or regenerate here
DRIVE_DIR = '/content/drive/MyDrive/Colab Notebooks'

X_test = torch.from_numpy(splits["X_test"]).float()
y_test = splits["y_test"]
snr_test = splits["snr_test"]

model.eval()
preds = []
with torch.no_grad():
    for i in range(0, len(X_test), 512):
        batch = X_test[i:i+512].to('cuda' if torch.cuda.is_available() else 'cpu')
        preds.append(model(batch).argmax(dim=1).cpu().numpy())
y_pred = np.concatenate(preds)

# All SNRs combined
plot_confusion_matrix_inline(y_test, y_pred, list(class_names),
    title="Stage 2 Confusion Matrix (all SNR)",
    save_path=f'{DRIVE_DIR}/stage2_confusion_matrix_all.png')

# High SNR only (>=10dB) — best-case view
plot_confusion_matrix_inline(y_test, y_pred, list(class_names),
    title="Stage 2 Confusion Matrix (SNR >= 10dB)",
    min_snr=10, snr=snr_test,
    save_path=f'{DRIVE_DIR}/stage2_confusion_matrix_high_snr.png')